# M02 · Figures and maps

**Week 1 · about 5.8 h (+ 1.5 h optional)** · before this: M01

> **한국어 요약** matplotlib의 그림(Figure)과 축(Axes)을 익히고 연구실 그림 규칙에 맞춰 그림을 그립니다. 지도 원의 지름에서 규모를 구하는 함수와 연도별 지진 수를 세는 함수를 만들고, 2019년 상주 지진의 시간-규모 그림과 PyGMT 지도를 그립니다. 그림에 나온 숫자는 모두 코드로 출력해 확인합니다.

**You will**

- Read a circle's diameter back as a magnitude
- Count earthquakes per UTC year, quiet years included
- Draw a time-magnitude figure in the house style
- Map the Sangju sequence and nearby stations with PyGMT

**New words:** figure and axes (그림과 축 영역), sentence case (첫 단어만 대문자로 쓰는 표기), histogram (히스토그램), time-magnitude plot (시간-규모 그림), earthquake sequence (지진 시퀀스), station (관측소), map projection (지도 투영법), scale bar (축척 막대) · [all terms](../../../docs/glossary.md)

**Optional reading:** Plotting and Programming in Python, Plotting; PyGMT tutorials, first figure, data points

Log any AI help in `work/ai_log/M02.md` ([rules](../../../docs/ai_policy.md)); pass criteria: [criteria/M02.yaml](../../../.sgtl/criteria/M02.yaml).

---

© 2026 Minseong Seo (SGTL, Seoul National University) and contributors · code MIT · text CC BY 4.0 · cite CITATION.cff

In [ ]:
# Parameters: the only cell with values you may change. Run it first, then the setup cell.
SEQUENCE = "sangju2019"                    # a named sequence from st.data.sequence
YEARS = (2018, 2022)                       # first and last year for the yearly counts (UTC)
STATION_RADIUS_KM = 100.0                  # show the stations within this distance of the sequence center (km)
MAP_REGION = [126.8, 129.4, 35.5, 37.5]    # map region: lon_min, lon_max, lat_min, lat_max (degrees); holds every event
MAP_SCALE_KM = 50                          # length of the scale bar on the map (km)
NEAR_RADIUS_KM = 20.0                      # stretch task: radius around the mainshock (km)
NEAR_YEARS = (2000, 2025)                  # stretch task: first and last year (UTC)
FIGURE_DIR = "figures"                     # your figure files go here, next to this notebook

In [ ]:
import sgtl_tutorial as st
S = st.init("M02")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pygmt  # import PyGMT after st.init, which points it to this environment's GMT library

Path(FIGURE_DIR).mkdir(exist_ok=True)

## Part 1 · A figure and its axes (about 30 min)

This module draws figures with **Matplotlib**, a Python plotting library, and maps with **PyGMT**, Python access
to the Generic Mapping Tools. Every figure follows the lab's **house style**, a short list of rules.

One habit runs through the module: **print every number your figure shows**, so that a reader can check the
picture against the data. `st.glossary.term` explains a word; task 1 needs these two:

In [ ]:
print(st.glossary.term("function"))
print(st.glossary.term("vectorization"))

Matplotlib builds a plot from two objects. A **figure** is the whole picture, the thing you save. An **axes** is
one plotting area inside it, with its own x and y axis and labels. `fig, ax = plt.subplots()` makes one of each.
Draw with `ax.plot`, `ax.scatter` or `ax.hist`; label with `ax.set_xlabel`, `ax.set_ylabel` and `ax.set_title`:

In [ ]:
# Made-up numbers: four days of water levels (not seismic data).
day = [1, 2, 3, 4]
level_m = [2.0, 3.5, 3.0, 3.2]

fig, ax = plt.subplots(figsize=(4, 2.5))  # one figure with one axes; the size is in inches
ax.plot(day, level_m, marker="o")         # a line plot with a marker at each point
ax.set_xlabel("Day of the experiment")
ax.set_ylabel("Water level (m)")
ax.set_title("A toy line plot")

# Print every number the figure shows:
print("axes in this figure:", len(fig.axes), "| x range:", ax.get_xlim(), "| highest level:", max(level_m), "m")

A line plot (`ax.plot`) joins the points in order. A **scatter plot** (`ax.scatter`) draws one marker per point.
A **histogram** (`ax.hist`) splits values into bins, such as 1.0 to 1.1, and draws one bar per bin. These
magnitudes are made up by `st.synth`:

In [ ]:
rng = np.random.default_rng(2)                            # a fixed seed: the same numbers on every run
toy_m = st.synth.gr_magnitudes(rng, 300, b=1.0, mc=1.0)   # 300 made-up magnitudes, all 1.0 or larger
edges = np.arange(0.95, toy_m.max() + 0.1, 0.1)           # bin edges halfway between the 0.1 steps

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 2.8))     # one figure, two axes side by side
ax1.scatter(np.arange(toy_m.size), toy_m, s=6)            # s is the marker AREA in points squared
ax1.set_xlabel("Event number")
ax1.set_ylabel(st.plot.ML_LABEL)
heights, _, _ = ax2.hist(toy_m, bins=edges, color="0.6", edgecolor="black")
ax2.set_xlabel(st.plot.ML_LABEL)
ax2.set_ylabel("Number of events")
fig.tight_layout()                                        # keeps labels of neighboring axes apart

# Print every number the figure shows:
print(f"{toy_m.size} magnitudes from {toy_m.min():.1f} to {toy_m.max():.1f}")
print(f"tallest bar: {heights.max():.0f} events in the bin from {edges[heights.argmax()]:.2f}")

**The house style.** Every lab figure follows these rules, so that figures from different people fit together.

1. **Sentence case** in labels, titles and legends: capitalize only the first word and names ("Origin time (UTC)").
2. American English: color, center, gray, meter.
3. Helvetica: `st.init` set it already, and prints a note when it uses a similar font instead.
4. All four frame lines around each axes.
5. Units in parentheses, as in "Depth (km)", and short titles of at most 40 characters.

The local magnitude is written $\mathit{M}_{\mathrm{L}}$ (italic M, roman L); `st.plot.ML_LABEL` holds it.

In [ ]:
print("fonts, in order of preference:", plt.rcParams["font.sans-serif"][:3])
print("frame lines drawn:", [s for s in ("left", "right", "top", "bottom") if plt.rcParams[f"axes.spines.{s}"]])
print("magnitude label:", st.plot.ML_LABEL)

## Part 2 · Magnitude as marker size, and back (about 50 min)

**Magnitude** gives the size of an earthquake on a logarithmic scale. On a lab map, an earthquake of local
magnitude $M$ is a circle of diameter

$$d = 0.025\,\sqrt{5\,e^{2M}}\ \mathrm{cm}$$

with $e \approx 2.718$: a bigger earthquake gets a bigger circle. `st.plot.mag_size_cm(M)` applies this house
rule. A map reader goes the other way, from a circle to a magnitude: the **inverse** of the rule, which undoes
it. The **natural logarithm** `np.log` undoes $e^{x}$ (`np.exp`); `np.log10` undoes $10^{x}$:

In [ ]:
y = np.array([-1.0, 0.0, 2.5])
print("y                 ->", y)
print("np.log(np.exp(y)) ->", np.log(np.exp(y)))        # the natural logarithm undoes e to a power
print("np.log10(10.0**y) ->", np.log10(10.0 ** y))      # the base-10 logarithm undoes 10 to a power
print("np.exp(np.log(4)) ->", np.exp(np.log(4.0)))      # and e to a power undoes the natural logarithm

Your function must work on one number and on a whole column at once. That is **vectorization**: one calculation
on a whole array instead of a loop over its items. NumPy functions such as `np.exp` and `np.log` work this way;
Python lists do not:

In [ ]:
a = np.array([1.0, 2.0, 3.0])
print("2 * array     ->", 2 * a)                     # element by element
print("np.exp(array) ->", np.exp(a))                 # element by element
print("2 * list      ->", 2 * [1.0, 2.0, 3.0])       # a list is REPEATED, not multiplied
print("np.asarray(list) * 2 ->", np.asarray([1.0, 2.0, 3.0]) * 2)

**Watch out:** Matplotlib's `ax.scatter(..., s=...)` takes the marker **area** in points squared, but PyGMT takes
the **diameter**, here in cm. For Matplotlib, convert a house-rule diameter with `s = (d_cm / 2.54 * 72) ** 2`.

### M02-T1 · Magnitude from marker size · core · about 30 min

Write `ml_from_marker_cm(size_cm)`, the inverse of the house rule: the local magnitude of a circle of diameter
`size_cm` (cm). Solve the rule for $M$ on paper first, one operation at a time. One number gives one number; a
list, an array or a Series gives one magnitude per diameter, all at once.

Example: `ml_from_marker_cm(1.0)` gives about 2.88.

Not allowed: `st.plot.mag_size_cm` or anything else from the course package inside the function.

In [ ]:
# Before you code: by how much does the magnitude change when a circle gets twice as wide?
st.predict("M02-T1", """

""")

In [ ]:
def ml_from_marker_cm(size_cm):
    """Local magnitude(s) of house-rule circles of diameter size_cm: the inverse of the house rule.

    size_cm : a number, list, NumPy array or pandas Series of diameters in cm (all > 0)
    returns : a float for one number; otherwise a NumPy array (or Series) of the same length.
              Exact: the house rule applied to the result gives size_cm back. Circles smaller than
              the ML 0 circle give magnitudes below 0.
    """
    # Step 1: turn the input into a NumPy array of floats (a list cannot do element-by-element math).
    # Step 2: on paper, solve d = 0.025 * sqrt(5 * e^(2M)) for M, undoing one operation at a time.
    # Step 3: write your result once, with NumPy functions, for the whole array.
    # Step 4: give back a plain float when a single number came in.
    # YOUR CODE HERE
    raise NotImplementedError("M02-T1: write your code here")

In [ ]:
st.check("M02-T1", func=ml_from_marker_cm)

**Explain:** Run the cell below: its first four circles double in width each time. Why does the magnitude grow by
the same step at every doubling?

In [ ]:
# Use it: read a magnitude key backwards, then go forward again with the house rule.
key_cm = np.array([0.25, 0.5, 1.0, 2.0, 3.0])          # circle diameters on a map key (cm)
key_ml = ml_from_marker_cm(key_cm)
for d_cm, m in zip(key_cm, key_ml):
    print(f"a {d_cm:.2f}-cm circle stands for M_L {m:.2f}")
back_cm = st.plot.mag_size_cm(key_ml)
print(f"largest difference after the round trip: {np.max(np.abs(back_cm - key_cm)):.1e} cm")

## Part 3 · Counting events per year (about 65 min)

The course comes with KMA's earthquake catalog. Its license, KOGL Type 1, lets you reuse it if you name the
source, so every figure made from it names KMA. Our example is the 2019 Sangju sequence. An **earthquake
sequence** is a group of related earthquakes close in space and time. Its largest event is the mainshock; the
smaller events after it are **aftershocks**. `st.data.sequence("sangju2019")` gives its events within 10 km,
2018-2022:

In [ ]:
sangju = st.data.sequence(SEQUENCE)
print(sangju.attrs["source"])
print(len(sangju), "events; columns:", list(sangju.columns))
sangju.head(3)

**Watch out:** the origin time comes twice. `origin_utc` is in UTC, the clock of this course; `origin_kst` is in
Korea Standard Time, UTC + 9 h. An event at 20:00 UTC on 31 December is already 1 January in Korea, so the two
clocks can disagree about the year.

A pandas Series is one column of values with labels, its index. Yearly counts fit a Series indexed by year. In
this made-up example, 2020 is simply absent:

In [ ]:
toy_counts = pd.Series([3, 1, 4], index=[2019, 2021, 2022])   # made-up counts
print(toy_counts)
print("index:", list(toy_counts.index), "| value for 2021:", toy_counts.loc[2021])
print("Is 2020 in the index?", 2020 in toy_counts.index)

A mean, a rate or a bar chart made from `toy_counts` would skip 2020 without a warning. A quiet year is
information: it must appear, with 0.

### M02-T2 · Events per year, zeros included · core · about 45 min

Write `yearly_counts(df, start_year, end_year)`. It returns a pandas Series indexed by the years from `start_year`
to `end_year`, both included. Each value is the number of events in that UTC year, 0 for a year without events.

Example: events at 2019-03-01, 2019-12-31 20:00 and 2021-06-01 (UTC) give, for the years 2018 to 2021, the counts
0, 2, 0 and 1.

In [ ]:
# Before you code: in which year from 2018 to 2022 do you expect the most Sangju events?
st.predict("M02-T2", """

""")

In [ ]:
def yearly_counts(df, start_year, end_year):
    """Number of events per UTC calendar year, for every year from start_year to end_year (both included).

    df : catalog DataFrame with a time-zone-aware column 'origin_utc' (not changed)
    start_year, end_year : int
    returns : pandas Series of integer counts indexed by the years (int, increasing, once each);
              0 for a year without events; events outside the range are not counted
    """
    # Step 1: the UTC year of every event.
    # Step 2: how many events each year has.
    # Step 3: put the counts onto the full list of years, with 0 for a year without events.
    # YOUR CODE HERE
    raise NotImplementedError("M02-T2: write your code here")

In [ ]:
st.check("M02-T2", func=yearly_counts)

**Explain:** Run the cell below and compare the counts with your prediction. What do they say about the
aftershocks of the ML 3.9 mainshock?

In [ ]:
# Use it: the Sangju counts, printed and drawn as a bar chart.
counts = yearly_counts(sangju, *YEARS)
print(counts.to_string())
print("total:", int(counts.sum()), "of", len(sangju), "events in the sequence")

fig, ax = plt.subplots(figsize=(4.5, 2.6))
ax.bar(counts.index, counts.to_numpy(), color="0.6", edgecolor="black")
ax.set_xticks(counts.index)
ax.set_xlabel("Year (UTC)")
ax.set_ylabel("Number of events")
ax.set_title("Sangju sequence per year")
fig.tight_layout()

## Part 4 · A time-magnitude figure (about 95 min)

A **time-magnitude plot** puts each event at its origin time (x) and its magnitude (y). It shows when a sequence
started, how big it got and how it went on. Give Matplotlib a column of times as x, and it picks the date ticks
itself. The tick labels are in UTC, so the x label should say "(UTC)". Made-up daily values:

In [ ]:
toy_days = pd.date_range("2021-03-01", periods=10, freq="D", tz="UTC")   # ten made-up days
toy_values = [3, 5, 4, 6, 8, 7, 7, 9, 6, 5]
fig, ax = plt.subplots(figsize=(4.5, 2.4))
ax.plot(toy_days, toy_values)
ax.set_xlabel("Date (UTC)")
ax.set_ylabel("Made-up value")
fig.autofmt_xdate()                       # tilts the date labels so they do not overlap
print("x data type:", toy_days.dtype, "| first and last day:", toy_days[0].date(), toy_days[-1].date())

A function that makes a figure should **return the Figure object**. Then the caller decides: show it, change it,
or save it with `fig.savefig("name.png")` (and `.pdf` for print). Saving or `plt.show()` inside the function
takes that choice away.

### M02-T3 · Time-magnitude figure · core · about 75 min

Write `time_magnitude_figure(df, title)`. It draws one marker per event at its origin time and magnitude and
returns the Matplotlib `Figure`. Follow the house style of Part 1. The x label names the time zone, the y label
uses `st.plot.ML_LABEL`, and `title` is the title. Here `st.plot` is allowed inside the function.

Example: `sangju.head(3)` gives three markers in 2018 (February, May, August) at ML 1.4, 1.2 and 1.2.

Not allowed: `plt.show()` inside the function.

In [ ]:
# Before you code: where on the Sangju figure will the tallest marker sit?
st.predict("M02-T3", """

""")

In [ ]:
def time_magnitude_figure(df, title):
    """Magnitude against origin time, one marker per event; returns the Matplotlib Figure.

    df      : catalog DataFrame with 'origin_utc' (time-zone-aware UTC) and 'ml'
    title   : short title in sentence case (at most 40 characters)
    returns : a matplotlib Figure with at least one axes; its x axis holds the origin_utc column itself
              as dates (not event numbers or decimal years), its y values are the magnitudes.
              Every label, title and legend entry in sentence case and American English (color, center,
              gray, meter); every title at most 40 characters; all four frame lines kept; no plt.show().
    """
    # Step 1: one figure with one axes.
    # Step 2: one marker per event at (origin time, magnitude).
    # Step 3: labels (sentence case, time zone, magnitude symbol) and the title; return the figure.
    # YOUR CODE HERE
    raise NotImplementedError("M02-T3: write your code here")

In [ ]:
st.check("M02-T3", func=time_magnitude_figure)

**Explain:** Run the cell below. Your x axis is in UTC: at what time and date in Korea (KST) did the largest
Sangju event happen?

In [ ]:
# Use it: the Sangju figure, saved as PNG and PDF.
fig = time_magnitude_figure(sangju, "Sangju sequence, 2018-2022")
for ext in ("png", "pdf"):
    fig.savefig(Path(FIGURE_DIR) / f"sangju_time_magnitude.{ext}")

# Print every number the figure shows:
big = sangju.loc[sangju["ml"].idxmax()]
first, last = sangju["origin_utc"].min(), sangju["origin_utc"].max()
print(f"{len(sangju)} events from {first:%Y-%m-%d} to {last:%Y-%m-%d} (UTC)")
print(f"M_L from {sangju['ml'].min():.1f} to {sangju['ml'].max():.1f}; "
      f"largest: event {big['event_id']} at {big['origin_utc']:%Y-%m-%d %H:%M:%S} UTC")
print("Source:", sangju.attrs["source"])

## Part 5 · Maps with PyGMT (about 110 min)

PyGMT draws a map in layers on one `pygmt.Figure`, and a later layer covers an earlier one. `fig.basemap` sets the
**map region**, `[lon_min, lon_max, lat_min, lat_max]` in degrees. It also sets the **map projection**, the rule
that flattens the Earth onto paper (`"M10c"`: Mercator, 10 cm wide). `fig.coast` draws the coast. A **scale bar**
is a bar of known length (`map_scale=...`): draw it after the land. Inside `with st.plot.gmt_style():` a map gets
decimal degrees, a plain frame and Helvetica:

In [ ]:
demo_region = [124.0, 132.0, 33.0, 39.0]                     # lon_min, lon_max, lat_min, lat_max (degrees)
with st.plot.gmt_style(FONT_TITLE="12p,Helvetica"):          # house style, plus a smaller title font
    fig = pygmt.Figure()
    fig.basemap(region=demo_region, projection="M10c", frame=["af", "+tToy map"])
    fig.coast(land="gray90", water="white", shorelines="0.25p,gray30")
    fig.basemap(map_scale=st.plot.scale_bar_spec(demo_region, km=100))   # after the land, so it stays visible
fig.show()
print("region:", demo_region, "| scale bar settings:", st.plot.scale_bar_spec(demo_region, km=100))

Symbols and labels on made-up points. `fig.plot(x=..., y=..., style=...)` draws symbols: `"c0.3c"` is a circle
0.3 cm wide, `"t0.3c"` a triangle. With `style="cc"`, each diameter in cm comes from `size=`. `fig.text(...,
justify="LM", offset="0.4c/0c")` writes a label beside its symbol. `pygmt.makecpt` and `fig.colorbar` add
colors; `fig.legend` lists the symbols drawn with `label=`:

In [ ]:
toy = pd.DataFrame({"name": ["P1", "P2", "P3"], "lon": [127.0, 128.6, 126.5], "lat": [37.5, 35.2, 33.4],
                    "value": [5.0, 12.0, 20.0], "size_cm": [0.2, 0.4, 0.6]})
with st.plot.gmt_style(FONT_TITLE="12p,Helvetica"):
    fig = pygmt.Figure()
    fig.basemap(region=demo_region, projection="M10c", frame=["af", "+tToy symbols"])
    fig.coast(land="gray90", water="white", shorelines="0.25p,gray30")
    pygmt.makecpt(cmap="viridis", series=[0, 25])            # colors for values from 0 to 25
    fig.plot(x=toy["lon"], y=toy["lat"], size=toy["size_cm"], style="cc", fill=toy["value"], cmap=True,
             pen="0.5p,black")
    fig.colorbar(frame=["af", "x+lMade-up value"], position="JMR+o0.5c/0c+w4c")
    fig.plot(x=toy["lon"], y=toy["lat"], style="x0.15c", pen="0.8p,red", label="Toy point")
    fig.text(x=toy["lon"], y=toy["lat"], text=toy["name"], justify="LM", offset="0.4c/0c", font="8p,Helvetica")
    fig.legend(position="jTL+o0.2c", box="+gwhite+p0.5p")
    fig.basemap(map_scale=st.plot.scale_bar_spec(demo_region, km=100))
fig.show()

# Print every number the map shows:
print(toy.to_string(index=False))

A **station** is a site with a seismometer, named by a network code (KS for KMA, KG for KIGAM) and a station code.
The cell keeps the stations within `STATION_RADIUS_KM` of the sequence center. It measures the **great-circle
distance**, the shortest way along the Earth's surface.

**Watch out:** the lab locates earthquakes with stations within about 100 km. Farther away, the first wave can be
a **head wave**, one that ran along the base of the crust. The lab never uses head waves for locations.

In [ ]:
from pyproj import Geod

center = sangju.attrs["sequence"]              # the sequence settings, including its center
stations_all = st.data.stations()
n_sta = len(stations_all)
_, _, dist_m = Geod(ellps="WGS84").inv(np.full(n_sta, center["center_lon"]), np.full(n_sta, center["center_lat"]),
                                       stations_all["longitude"].to_numpy(), stations_all["latitude"].to_numpy())
near_stations = stations_all.loc[dist_m / 1000.0 <= STATION_RADIUS_KM].reset_index(drop=True)
print(f"{len(near_stations)} of {n_sta} stations lie within {STATION_RADIUS_KM:g} km of "
      f"{center['center_lat']:.2f} N, {center['center_lon']:.2f} E")
print(near_stations["network"].value_counts().to_string())

### M02-T4 · Map of the Sangju sequence · core · about 100 min

Write `map_events(df, stations, region, scale_km)`, a lab-style map; it returns the `pygmt.Figure`. Draw inside
`with st.plot.gmt_style():`, with exactly `region`, a short title and a `map_scale` bar `scale_km` long, after the
land. Events are circles: `fig.plot(..., size=..., style="cc")` with one house-rule diameter per event
(`st.plot.mag_size_cm`, allowed here). Every station gets a symbol (`fig.plot`) and its code beside it
(`fig.text`).

Example: an ML 2.0 event gets a circle 0.41 cm wide.

Not allowed: `fig.show()` inside the function.

In [ ]:
# Before you code: on your map, will the circle of the ML 3.9 mainshock cover the whole Sangju cluster?
st.predict("M02-T4", """

""")

In [ ]:
def map_events(df, stations, region, scale_km):
    """PyGMT map of earthquakes (circles sized by magnitude) and stations (labeled symbols).

    df       : catalog DataFrame with longitude, latitude (degrees), ml and depth_km
    stations : DataFrame with station, longitude, latitude
    region   : [lon_min, lon_max, lat_min, lat_max] in degrees; the map uses exactly this region
    scale_km : length of the scale bar in km (map_scale of fig.basemap or fig.coast)
    returns  : the pygmt.Figure (fig.show() is not called)
    """
    # Step 1: inside "with st.plot.gmt_style():", a new figure with a frame for exactly `region` and a short title.
    # Step 2: the events as circles (style "cc"), sized by one house-rule diameter in cm per event.
    # Step 3: every station as a symbol, with its code written beside it.
    # Step 4: a scale bar scale_km long, drawn after the land; return the figure (no fig.show() inside).
    # YOUR CODE HERE
    raise NotImplementedError("M02-T4: write your code here")

In [ ]:
st.check("M02-T4", func=map_events, region=MAP_REGION, scale_km=MAP_SCALE_KM)

**Explain:** Run the cell below. Does the ML 3.9 circle hide the small events on your map? Use the printed
circle sizes.

In [ ]:
# Use it: the Sangju map, shown and saved.
fig = map_events(sangju, near_stations, MAP_REGION, MAP_SCALE_KM)
fig.show()
for ext in ("png", "pdf"):
    fig.savefig(Path(FIGURE_DIR) / f"sangju_map.{ext}")

# Print every number the map shows:
sizes_cm = np.sort(np.asarray(st.plot.mag_size_cm(sangju["ml"]), dtype=float))
print("region (lon_min, lon_max, lat_min, lat_max):", MAP_REGION, "| scale bar:", MAP_SCALE_KM, "km")
print(f"{len(sangju)} events, M_L {sangju['ml'].min():.1f} to {sangju['ml'].max():.1f}, "
      f"depth {sangju['depth_km'].min():.0f} to {sangju['depth_km'].max():.0f} km")
print(f"circle diameters from {sizes_cm[0]:.2f} to {sizes_cm[-1]:.2f} cm; read back with your inverse: "
      f"M_L {ml_from_marker_cm(sizes_cm[0]):.1f} to {ml_from_marker_cm(sizes_cm[-1]):.1f}")
print(len(near_stations), "stations within", STATION_RADIUS_KM, "km | source:", sangju.attrs["source"])

## Part 6 · Stretch: how many events near the mainshock? (about 90 min)

`st.data.sequence` cut the Sangju sequence out with a fixed 10-km circle. Is that a good circle? Count the events
of the whole catalog in a circle of your own radius around the mainshock, year by year. The counts mix the
sequence with the **seismicity**, the earthquakes of the region over the years, that was there before it.

### M02-T5 · Yearly counts near the mainshock · stretch · about 90 min

Write `events_near(df, lat0, lon0, radius_km)`: the rows whose **epicenter** lies at most `radius_km` from
`(lat0, lon0)`. The epicenter is the point on the surface above the start of the rupture. Use the great-circle
distance on a sphere of radius 6371 km. Then write `counts_near(df, lat0, lon0, radius_km, start_year,
end_year)`, their counts per year like `yearly_counts`.

Example: a point 0.1° north of `(lat0, lon0)` is 11.1 km away: kept for `radius_km` = 12, not for 11.

Not allowed: the course package inside the functions.

In [ ]:
# Before you code: within NEAR_RADIUS_KM of the mainshock, will 2019 stand out among the years 2000-2025?
st.predict("M02-T5", """

""")

In [ ]:
def events_near(df, lat0, lon0, radius_km):
    """Rows of df whose epicenter lies at most radius_km from the point (lat0, lon0).

    df         : catalog DataFrame with latitude and longitude in degrees
    lat0, lon0 : the point, in degrees
    radius_km  : float, km; great-circle distance on a sphere of radius 6371 km, kept when <= radius_km
    returns    : the kept rows of df, all columns
    """
    # YOUR CODE HERE
    raise NotImplementedError("M02-T5: write your code here")


def counts_near(df, lat0, lon0, radius_km, start_year, end_year):
    """Events per UTC year within radius_km of (lat0, lon0): a pandas Series like yearly_counts gives,
    indexed by every year from start_year to end_year (int, increasing), zeros included."""
    # YOUR CODE HERE
    raise NotImplementedError("M02-T5: write your code here")

In [ ]:
st.check("M02-T5", func=counts_near)

**Explain:** Run the cell below. Does 2019 stand out in the printed counts? Say what that means for the events
around Sangju.

In [ ]:
# Use it: counts and a time-magnitude figure around the mainshock.
catalog = st.data.kma_catalog()
main = sangju.loc[sangju["ml"].idxmax()]
print(f"Mainshock {main['event_id']}: {main['origin_utc']:%Y-%m-%d %H:%M:%S} UTC, M_L {main['ml']:.1f}, "
      f"{main['latitude']:.2f} N, {main['longitude']:.2f} E, depth {main['depth_km']:.0f} km")
near_counts = counts_near(catalog, main["latitude"], main["longitude"], NEAR_RADIUS_KM, *NEAR_YEARS)
print(near_counts.to_string())

near = events_near(catalog, main["latitude"], main["longitude"], NEAR_RADIUS_KM)
t0 = pd.Timestamp(year=NEAR_YEARS[0], month=1, day=1, tz="UTC")
t1 = pd.Timestamp(year=NEAR_YEARS[1] + 1, month=1, day=1, tz="UTC")
near = near[(near["origin_utc"] >= t0) & (near["origin_utc"] < t1)]
fig = time_magnitude_figure(near, f"Within {NEAR_RADIUS_KM:g} km of the mainshock")
for ext in ("png", "pdf"):
    fig.savefig(Path(FIGURE_DIR) / f"sangju_within_{NEAR_RADIUS_KM:g}km.{ext}")
print(f"{len(near)} events, M_L {near['ml'].min():.1f} to {near['ml'].max():.1f} | source: {catalog.attrs['source']}")

## Wrap-up

Your mentor reviews your figures with the [rubric](../../../docs/curriculum.md#gates). It asks for the house style,
units and time zones in labels, and a scale bar on maps. Every number of a figure is printed next to it, and KMA
is named as the source. Your progress in this module:

In [ ]:
st.progress("M02")

**Read more:** Rougier, Droettboom and Bourne (2014), ten rules for better figures; Crameri, Shephard and Heron
(2020), color in science; Wessel et al. (2019), GMT 6. The [reading list](../../../docs/reading_list.md) has the rest.

Next: M04 (earthquake catalogs). Commit your notebook as in M00.